In [0]:
from pyspark.sql.functions import *

In [0]:
source_path = "/Volumes/finguard/source/fraud_watchlist/source_data/"
checkpoint_path = "/Volumes/finguard/source/fraud_watchlist/checkpoint/"
schema_path = "/Volumes/finguard/source/fraud_watchlist/schema/"

In [0]:
dbutils.fs.ls(source_path)

[FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174815_666089_0.json', name='fraud_watchlist_20260810_174815_666089_0.json', size=365, modificationTime=1786384097000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174821_429872_1.json', name='fraud_watchlist_20260810_174821_429872_1.json', size=379, modificationTime=1786384102000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174826_849761_2.json', name='fraud_watchlist_20260810_174826_849761_2.json', size=380, modificationTime=1786384108000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174832_477182_3.json', name='fraud_watchlist_20260810_174832_477182_3.json', size=383, modificationTime=1786384113000),
 FileInfo(path='dbfs:/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174837_865119_4.json', na

In [0]:
input_steam = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("pathGlobFilter", "*.json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .load(source_path)
)

In [0]:
transformed_df = input_steam.select(
    "*",
    col("_metadata.file_path").alias("file_path"),
    current_timestamp().alias("ingestion_timestamp"),
)

In [0]:
streaming_query = (
    transformed_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("finguard.bronze.fraud_watchlist_test")
)

In [0]:
%sql
select * from finguard.bronze.fraud_watchlist

watchlist_id,watch_type,entity_id,risk_level,action,reason_code,reason_description,status,effective_from,reported_by,reported_source,city,country,_rescued_data,source_file,ingestion_timestamp
wl000006,CARD,4009109391119268,high,ADD,TEST_TRANSACTION,Auto-generated fraud watchlist entry,ACTIVE,18-Jun-2026 09:12:00,SOC,Manual,Mumbai,India,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174843_348226_5.json,2026-08-10T18:36:13.380Z
wl000005,CARD,4260650191792026,critical,ADD,ACCOUNT_TAKEOVER,Auto-generated fraud watchlist entry,ACTIVE,18-Jun-2026 09:07:00,Customer Care,CRM,Bengaluru,India,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174837_865119_4.json,2026-08-10T18:36:13.380Z
wl000004,CARD,4717327980595032,high,ADD,TEST_TRANSACTION,Auto-generated fraud watchlist entry,ACTIVE,18-Jun-2026 09:04:00,AML Team,External Feed,Chennai,India,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174832_477182_3.json,2026-08-10T18:36:13.380Z
wl000003,CARD,5329839542961842,low,ADD,TEST_TRANSACTION,Auto-generated fraud watchlist entry,ACTIVE,18-Jun-2026 09:02:00,AML Team,External Feed,Delhi,India,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174826_849761_2.json,2026-08-10T18:36:13.380Z
wl000002,CARD,4413337312552355,high,ADD,BLACKLISTED_MERCHANT,Auto-generated fraud watchlist entry,ACTIVE,18-Jun-2026 09:01:00,SOC,External Feed,Pune,India,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174821_429872_1.json,2026-08-10T18:36:13.380Z
wl000001,CARD,5008514036965665,high,ADD,PHISHING,Auto-generated fraud watchlist entry,ACTIVE,18-Jun-2026 09:00:00,SOC,Manual,Hyderabad,India,null,/Volumes/finguard/source/fraud_watchlist/source_data/fraud_watchlist_20260810_174815_666089_0.json,2026-08-10T18:36:13.380Z


In [0]:
%sql 
-- drop table if exists finguard.bronze.fraud_watchlist